In [ ]:
"""
Lymphoma CAR-T single-cell: moscot temporal optimal-transport analysis (T/NK).

Computes cell-type transition maps across timepoints, pulls ancestor mass for
selected cell types and splits pull mass by response group

Assumed to exist from earlier in the pipeline / project:
  - `get_celltype_name(ct)`   : maps a cell-type label to a filename-safe token.
  - `get_timepoint_name(t)`   : maps a timepoint tuple to a filename-safe token.

Timepoint codes used below: 2 = TDN (Product), 4 = Peak, 5 = Week 4.
"""


In [ ]:
# adopted from https://moscot.readthedocs.io/en/latest/notebooks/tutorials/200_temporal_problem.html

import os
import warnings

import moscot as mt
import moscot.plotting as mpl
from moscot.problems.time import TemporalProblem

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc

plt.rcParams["legend.scatterpoints"] = 1
sc.set_figure_params(scanpy=True, dpi=80, dpi_save=300)

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [ ]:
# set wd
os.chdir(".")

In [ ]:
# ---- CONFIG: directories ---------------------------------------------------
OBJECTS_DIR = "./python_objects"        # serialized TemporalProblem pickles
IMAGES_DIR = "./images"                 # figure outputs
RESULTS_DIR = "./files/moscot_results"  # driver-gene CSV outputs

for _d in (OBJECTS_DIR, IMAGES_DIR, RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)


# Example cell transition, pull distribution, and driver gene analysis for one Temporal Problem. 
All analysis for monocytes and non-CAR cells follows the same pattern, adjusted for specific cell types or time points

In [ ]:
def get_timepoint_name(x):
    if x==(2,4):
        return"_tdn_peak_"
    if x == (4,5):
        return"_peak_week4_"
    if x == (2,5):
        return"_global_"
    if x == (1,3):
        return"_aph_d0_"
    if x == (3,4):
        return "_d0_peak_"
    if x == (3,4,5):
        return "_d0_week4_"
    

In [ ]:
#Load temporal problem
tp2 = TemporalProblem.load(
    os.path.join(OBJECTS_DIR, "moscot_temporalproblem_CAROnly_TNK.pickle")
)

tp5 = TemporalProblem.load(
    os.path.join(OBJECTS_DIR, "moscot_temporalproblem_nonCAR_T-P-W_TNK.pickle"))

In [ ]:
# Cell Transitions
tp2.cell_transition(
    source=2,
    target=4,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_tdn_peak",
)

tp2.cell_transition(
    source=4,
    target=5,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_peak_4W",
)

tp2.cell_transition(
    source=2,
    target=5,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_tdn_4W",
)

#optional
tp2.save(os.path.join(OBJECTS_DIR, "moscot_temporalproblem_CAROnly_TNK_progress.pickle"))

# Plot
keys = ["CT_tdn_peak", "CT_peak_4W", "CT_tdn_4W"]

for k in keys:
    plt.figure(figsize=(20, 20))
    fig = mpl.cell_transition(tp2.adata, dpi=300, fontsize=8, key=k)
    plt.savefig(
        os.path.join(IMAGES_DIR, f"moscot_CAR_TNK_{k}.pdf"),
        format="pdf",
        bbox_inches="tight",
    )
    plt.close(fig)

In [ ]:
# Cell Transitions Non-CAR
tp5.cell_transition(
    source=2,
    target=4,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_tdn_peak",
)

tp5.cell_transition(
    source=4,
    target=5,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_peak_4W",
)

tp5.cell_transition(
    source=2,
    target=5,
    source_groups="cell.anno",
    target_groups="cell.anno",
    forward=True,
    key_added="CT_tdn_4W",
)

#optional
tp5.save(os.path.join(OBJECTS_DIR, "moscot_temporalproblem_nonCAR_T-P-W_TNK_progress.pickle"))

# Plot
keys = ["CT_tdn_peak", "CT_peak_4W", "CT_tdn_4W"]

for k in keys:
    plt.figure(figsize=(20, 20))
    fig = mpl.cell_transition(tp2.adata, dpi=300, fontsize=8, key=k)
    plt.savefig(
        os.path.join(IMAGES_DIR, f"moscot_nonCAR_T-P-W_TNK_{k}.pdf"),
        format="pdf",
        bbox_inches="tight",
    )
    plt.close(fig)

In [ ]:
#Pull Distrubition 

# Pulls using cell annotations
tp2.pull(source=2, target=5, data="cell.anno", subset="CD8 TEM", key_added="CD8_TEM_tdn_w4")
tp2.pull(source=4, target=5, data="cell.anno", subset="CD8 TEM", key_added="CD8_TEM_peak_w4")
tp2.pull(source=2, target=4, data="cell.anno", subset="CD8 TEM", key_added="CD8_TEM_tdn_peak")


# optional save progress
tp2.save(os.path.join(OBJECTS_DIR, "moscot_temporalproblem_CAROnly_TNK_progress.pickle"), overwrite=True)

In [ ]:
# get pull distribution per celltype
celltypes = ["CD8 TEM"]

for ct in celltypes:
    key_name = "CD8 TEM"

    tp2.pull(source=2, target=4, data="cell.anno", subset=ct,
             key_added="pull_" + key_name + "_tdn_peak")
    tp2.pull(source=4, target=5, data="cell.anno", subset=ct,
             key_added="pull_" + key_name + "_peak_week4")
    tp2.pull(source=2, target=5, data="cell.anno", subset=ct,
             key_added="pull_" + key_name + "_tdn_week4")

In [ ]:
# Sum pull mass over the consecutive transitions (TDN->Peak + Peak->W4) -
for ct in celltypes:
    key_name = "CD8 TEM"

    tp2.adata.obs["pull_" + key_name + "_tdn_peak"].fillna(0.00, inplace=True)
    tp2.adata.obs["pull_" + key_name + "_peak_week4"].fillna(0.000, inplace=True)

    tp2.adata.obs["pull_sum_" + key_name] = (
        tp2.adata.obs["pull_" + key_name + "_tdn_peak"]
        + tp2.adata.obs["pull_" + key_name + "_peak_week4"]
    )

# pull distribution visualization

In [ ]:
# check pull distribution range
tp2.adata.obs["pull_CD8_TEM_tdn_peak"].describe()

# add colors
names2 = [
    "CD4 TEM-like", "CD4 Naive",
    "CD8 TEM", "CD8 Naive", "Proliferating",
    "gdT", "MAIT", "NK", "Treg"
]

final2_colors = [
    "steelblue3", "lightskyblue",
    "darkgreen", "darkolivegreen2", "gray",
    "pink1", "red", "gold", "plum2"
]


tp2.adata.uns["cell.anno_colors"] = final2_colors

tp2.adata.obs["cell.anno"].cat.categories
len(tp2.adata.uns["cell.anno_colors"])

In [ ]:
#scale my own data
pullkey = 'pull_CD8_TEM_tdn_peak'
vals = tp2.adata.obs[pullkey].copy()

vmin, vmax = np.percentile(vals, [1, 99])

tp2.adata.obs[pullkey + "_scaled"] = np.clip(vals, vmin, vmax)

print(vals.describe())

print(vmin, vmax)

#add as uns. 
tp2.adata.uns["moscot_results"]["pull"][pullkey + "_scaled"] = tp2.adata.uns["moscot_results"]["pull"][pullkey]
tp2.adata.uns["moscot_results"]["pull"][pullkey + "_scaled"]

In [ ]:
# print to PDF
fig, axes = plt.subplots(ncols=2, figsize=(16, 6))
mpl.pull(
    tp2,
    time_points=[4],
    basis="X_umap",
    ax=axes[0],
    key= pullkey + "_scaled",
    title=["CD8 TEM at Peak"]
    
)

mpl.pull(
    tp2,
    time_points=[2],
    basis="X_umap",
    ax=axes[1],
    scale=False,
    title=["CD8 TEM Peak Ancestors (at Product)"],
    key=pullkey + "_scaled"
)

fig.subplots_adjust(wspace=0.3)
fig.savefig("CAR_cd8_em_pull_plots_tdn_to_peak.pdf", bbox_inches="tight")